In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
import json
from joblib import Parallel, delayed

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.cm as cm
import matplotlib.colors as mcolors

import seaborn as sns

import nibabel as nib

import nilearn.plotting as plotting

from scipy.stats import pearsonr, ttest_ind, mannwhitneyu, zscore, sem, f_oneway, permutation_test

import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import fdrcorrection
from statsmodels.duration.hazard_regression import PHReg

from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival
from sksurv.linear_model import CoxPHSurvivalAnalysis

from lifelines.statistics import proportional_hazard_test
from lifelines import CoxPHFitter
from lifelines.utils.lowess import lowess

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

import statsmodels.formula.api as smf

## Relative imports
from utils import rewrite_subjectID, process_data, correct_OS, load_maps


## Directories and paths
voxel_res = 1.5
root =  "/home/joan/Desktop/PROJECTS/Glioblastomas" # "D:\JoanFR_Sano"
os.makedirs(root, exist_ok=True)
MNI_DIR = "/home/joan/Documents/MNI_ICBM_2009b_NLIN_ASYM" # "C:/Users/user/Documents/Data/MNI"
atlas_lobes = f"/home/joan/Documents/Parcellation/USCLobesMNI-ICBM-2009b-NLIN-ASYM/BCI-DNI_brain.labelxMNI-ICBM-2009b-NLIN-ASYM__Warped.nii.gz"
lobe_labels = f"/home/joan/Documents/Parcellation/USCLobesMNI-ICBM-2009b-NLIN-ASYM/BCI-DNI_brain.labels.txt"
atlas_wm = f"/home/joan/Documents/Parcellation/XTRACT_MNI-ICBM-2009b-NLIN-ASYM/xtract-tract-atlases-maxprob5-1mmxMNI-ICBM-2009b-NLIN-ASYM__Warped.nii.gz"
WM_labels = f"/home/joan/Documents/Parcellation/XTRACT_MNI-ICBM-2009b-NLIN-ASYM/XTRACT.labels.txt"
T1_file = os.path.join(MNI_DIR, f"T1_{voxel_res}mm.nii")
T1 = nib.load(T1_file)
T1_brain = nib.load(os.path.join(MNI_DIR, f"T1_{voxel_res}mm_brain_mask.nii"))

Ni, Nj, Nk = T1.get_fdata().shape
ucsf_dir = "Glioblastoma_UCSF-PDGM_v3-20230111"
upenn_dir = "Glioblastoma_UPENN-GBM_v2-20221024"
tissue = "whole"
tdmap = "TDMap-lesion"

## Other
fmt = "svg"
dpi = 300
stream_th = 0
daysXmonth = 365/12
voxel_size = (0.5**3) * (1/1000) # 0.5 (mm³/voxel) X 0.001 (cm³/mm³)   
sites = [0, 1]
n_jobs = 16
nthreads = 2

## Load and process clinical data
DATA = correct_OS(
    process_data(
        root,
        stream_th,
        voxel_size,
        UCSF_dir=ucsf_dir,
        UPENN_dir=upenn_dir,
        z_transform=0,
        cohort_ids=sites
    ),
    cohort_ids=sites,
    n_perms=10
)

# Process and load UCSF maps
ucsf_ids = [rewrite_subjectID(id) for id in DATA.loc[DATA["site"]==0]["ID"].values]
results = Parallel(n_jobs=n_jobs)(
    delayed(load_maps)(
        list_of_subjects=[subj_id],
        nFeatures=Ni * Nj * Nk,
        template=T1_file,
        root=root,
        data_dir=f"{ucsf_dir}/TDMaps_Grade-IV",
        tissue=tissue,
        tdmap=tdmap,
        voxel_res=voxel_res,
        nthreads=nthreads
    ) for subj_id in tqdm(ucsf_ids, desc="Loading UCSF maps")
)
maps_ucsf = np.vstack(results) # Nsubjects x nFeatures
maps_ucsf = np.where(maps_ucsf>=0, maps_ucsf, 0) # The resampling has introduced some small amount of negative numbers ## Be careful and double check
 
# Process and load UPENN maps
upenn_ids = DATA.loc[DATA["site"]==1]["ID"].values
results = Parallel(n_jobs=n_jobs)(
    delayed(load_maps)(
        list_of_subjects=[subj_id],
        nFeatures=Ni * Nj * Nk,
        template=T1_file,
        root=root,
        data_dir=f"{upenn_dir}/TDMaps_IDH1-WT",
        tissue=tissue,
        tdmap=tdmap,
        voxel_res=voxel_res,
        nthreads=nthreads
    ) for subj_id in tqdm(upenn_ids, desc="Loading UPENN maps")
)
maps_upenn = np.vstack(results) # Nsubjects x nFeatures
maps_upenn = np.where(maps_upenn>=0, maps_upenn, 0) # The resampling has introduced some small amount of negative numbers ## Be careful and double check

maps_all = np.vstack((maps_ucsf, maps_upenn))

In [ ]:
# Load CORRECTED survival data
os_all = DATA["OS (corrected)"] / daysXmonth
status_all = DATA["status"] 

os_ucsf = DATA.loc[DATA["site"]==0]["OS (corrected)"] / daysXmonth
status_ucsf = DATA.loc[DATA["site"]==0]["status"] 

os_upenn = DATA.loc[DATA["site"]==1]["OS (corrected)"] / daysXmonth
status_upenn = DATA.loc[DATA["site"]==1]["status"] 


In [ ]:
from TractDensityComponents import TD_PCA
from components import Projections

In [ ]:
# Normalization and masking strategies
normalization = None
masking_strategy = True


# Directories to save the results and outputs
results_root = os.path.join(root,f"{ucsf_dir}/Figures/TDMaps_Grade-IV/PCA")
os.makedirs(results_root, exist_ok=True)
save_dir = os.path.join(results_root, f"PCA-TD_Norm-{normalization}_Train-UCSF_Mask-{masking_strategy}")
os.makedirs(save_dir, exist_ok=True)

# Create the PCA object 
TD_pca = TD_PCA(
    training_maps=maps_ucsf, # UCSF as training
    template=T1,             # 1.5 mm template on MNI
    binary_mask=T1_brain,    # 1.5 mm brain mask on MNI to speed up the calculations
    n_components=120         # Number of components to fit
)
# Fit the PCA on the training data
cmps_ucsf = TD_pca.fit_pca_maps(
    norm=normalization       # Determine the normalization strategy
)

# Transform the UPENN maps (i.e., test) under the same conditions
cmps_upenn = TD_pca.transform_maps(
    maps=maps_upenn         
)

### Reconstructions and survival

In [ ]:
explained_all, corrs_all, cosine_sim = np.load("../explained_variance.npy").T, np.load("../correlations.npy").T, np.load("../cosine_sim.npy").T

In [ ]:
os_explained_ucsf = np.zeros((100,2))
os_similarity_ucsf = np.zeros((100,2))
os_cosine_ucsf = np.zeros((100,2))

n_perms = 5000
for k in tqdm(range(100)):
    result = permutation_test(
        (os_ucsf[status_ucsf==1], explained_all[status_ucsf==1, k]),
        statistic=lambda x, y: pearsonr(x,y)[0],
        permutation_type='pairings', # scipy docs says this is the appropriate one for correlations
        n_resamples=n_perms,
        alternative='two-sided', # Although we could easily justify a one sided test from the conceptual point of view
    )
    os_explained_ucsf[k, 0] = result.statistic
    os_explained_ucsf[k, 1] = result.pvalue

    result = permutation_test(
        (os_ucsf[status_ucsf==1], corrs_all[status_ucsf==1, k]),
        statistic=lambda x, y: pearsonr(x,y)[0],
        permutation_type='pairings', # scipy docs says this is the appropriate one for correlations
        n_resamples=n_perms,
        alternative='two-sided', # Although we could easily justify a one sided test from the conceptual point of view
    )
    os_similarity_ucsf[k, 0] = result.statistic
    os_similarity_ucsf[k, 1] = result.pvalue

    result = permutation_test(
        (os_ucsf[status_ucsf==1], cosine_sim[status_ucsf==1, k]),
        statistic=lambda x, y: pearsonr(x,y)[0],
        permutation_type='pairings', # scipy docs says this is the appropriate one for correlations
        n_resamples=n_perms,
        alternative='two-sided', # Although we could easily justify a one sided test from the conceptual point of view
    )
    os_cosine_ucsf[k, 0] = result.statistic
    os_cosine_ucsf[k, 1] = result.pvalue

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(os_explained[:,0])
ax[1].plot(np.log(os_explained[:,1]))
ax[1].hlines(np.log(0.05), 0, 100, 'r', '--')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(os_similarity[:,0])
ax[1].plot(np.log(os_similarity[:,1]))
ax[1].hlines(np.log(0.05), 0, 100, 'r', '--')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(os_cosine[:,0])
ax[1].plot(np.log(os_cosine[:,1]))
ax[1].hlines(np.log(0.05), 0, 100, 'r', '--')

In [ ]:
plt.hist(os[status==1])

In [ ]:
img = unflatten_maps(low_rank_ucsf[0], template=T1, make_nan=True, return_nifti=True)
nib.save(img, "../reconstruction_1.nii.gz")

In [ ]:
img = unflatten_maps(error.mean(axis=0), template=T1, make_nan=True, return_nifti=True)
nib.save(img, "../reconstruction_error_100.nii.gz")

In [ ]:
os = DATA["OS (corrected)"] / daysXmonth
status = DATA["status"] 

os_scores = np.zeros((100,2))
n_perms = 5000
for k in tqdm(range(100)):
    result = permutation_test(
        (os[status==1], cmps_ucsf[status==1, k]),
        statistic=lambda x, y: pearsonr(x,y)[0],
        permutation_type='pairings', # scipy docs says this is the appropriate one for correlations
        n_resamples=n_perms,
        alternative='two-sided', # Although we could easily justify a one sided test from the conceptual point of view
    )
    os_scores[k, 0] = result.statistic
    os_scores[k, 1] = result.pvalue

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(os_scores[:,0])
ax[1].plot(np.log(os_scores[:,1]))
ax[1].hlines(np.log(0.05), 0, 100, 'r', '--')

# Predicting survival with specific "anatomical modes"

In [ ]:
os = DATA["OS (corrected)"] / daysXmonth
status = DATA["status"] 

n_perms = 5000

In [ ]:
def rank_components_by_survival(
    cmps,
    os,
    status,
    transformation=lambda s: np.abs(s),
    n_components=None,
    n_perms=2000,
    random_state=0,
):
    """
    Rank PCA components by absolute association with survival.

    Parameters
    ----------
    cmps : ndarray (n_subjects, n_components_total)
        Component scores.
    os : array-like (n_subjects,)
        Survival time (continuous).
    status : array-like (n_subjects,)
        Event indicator (1 event observed, 0 censored).
    transformation : callable, default=abs
        Function applied to component scores before correlation.
    n_components : int or None
        Number of components to evaluate (defaults to cmps.shape[1]).
    n_perms : int
        Number of permutations when using permutation test.
    random_state : int
        Seed for permutation_test.

    Returns
    -------
    order : ndarray of ints, shape (n_eval_components,)
        Indices of components sorted by strength (strongest first).
    stats : DataFrame with columns ['r', 'p_perm']
    """
    cmps = np.asarray(cmps)
    _, n_total = cmps.shape
    if n_components is None:
        n_components = n_total

    mask_unc = np.asarray(status) == 1
    os_unc = np.asarray(os)[mask_unc]

    rs = np.zeros(n_components)
    p_perm = np.full(n_components, np.nan)

    for k in range(n_components):
        scores_k = transformation(cmps[mask_unc, k])

        # permutation_test returns an object with .statistic and .pvalue
        res = permutation_test(
            (os_unc, scores_k),
            statistic=lambda x, y: pearsonr(x, y)[0],
            permutation_type="pairings",
            n_resamples=n_perms,
            alternative="two-sided",
            random_state=random_state,
            vectorized=False,
        )
        rs[k] = res.statistic
        p_perm[k] = res.pvalue

    stats = pd.DataFrame({"r": rs, "p_perm": p_perm})
    order = np.argsort(stats["p_perm"].values)  # smallest p first

    return order, stats

In [ ]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.calibration import survival_probability_calibration

def evaluate_topk_models(
    cmps,
    os,
    status,
    order,
    max_k=10,
    transformation=lambda s: np.abs(s),
    return_models=False,
):
    """
    For k=1..max_k use the top-k components (from `order`) to build:
     - OLS on uncensored subjects (predict OS)
     - Cox PH on all subjects (duration=os, status=status)

    Parameters
    ----------
    cmps : ndarray (n_subjects, n_components_total)
    os : array-like (n_subjects,)
    status : array-like (n_subjects,)
    order : array-like
        Component indices sorted from strongest -> weakest.
    max_k : int
        Maximum k to evaluate (k will be min(max_k, len(order))).
    transformation : callable
        Applied to component scores before model building (kept for consistency).
    return_models : bool
        If True, return fitted model objects in output dictionary (can be large).

    Returns
    -------
    results : dict with keys:
        'k_list' : list of k values
        'ols' : list of dicts with keys ['r2', 'r2_adj', 'f_pvalue', 'params', 'summary_str']
        'cox' : list of dicts with keys ['loglike', 'aic', 'params', 'summary_str']
        optionally if return_models True: 'ols_models' and 'cox_models' lists
    """
    cmps = np.asarray(cmps)
    n_subj, n_total = cmps.shape
    k_max = min(max_k, len(order))

    results = {"k_list": list(range(1, k_max + 1)), "ols": [], "cox": []}
    if return_models:
        results["ols_models"] = []
        results["cox_models"] = []

    mask_unc = np.asarray(status) == 1
    os_arr = np.asarray(os)
    status_arr = np.asarray(status)

    for idx_k, k in enumerate(range(1, k_max + 1)):
        top_idx = order[:k]
        X_all = transformation(cmps[:, top_idx])  # all subjects for Cox
        X_unc = X_all[mask_unc, :]
        y_unc = os_arr[mask_unc]

        # ---------- OLS on uncensored ----------
        X_ols = sm.add_constant(X_unc, has_constant="add")
        ols_model = sm.OLS(y_unc, X_ols).fit()
        ols_res = {
            "r2": float(ols_model.rsquared),
            "r2_adj": float(ols_model.rsquared_adj),
            "f_pvalue": float(ols_model.f_pvalue) if hasattr(ols_model, "f_pvalue") else np.nan,
            "params": ols_model.params,
            "summary_str": ols_model.summary().as_text(),
        }
        results["ols"].append(ols_res)
        if return_models:
            results["ols_models"].append(ols_model)

        # ---------- Cox PH (lifelines) ----------
        df = pd.DataFrame(X_all, columns=[f"PC{i+1}" for i in range(k)])
        df["OS"] = os_arr
        df["status"] = status_arr

        cph = CoxPHFitter()
        cph.fit(df, duration_col="OS", event_col="status", show_progress=False)

        # C-index
        cindex = concordance_index(os_arr, -cph.predict_partial_hazard(df), status_arr)

        # Brier score: use lifelines calibration utils
        # Example: compute at median OS
        median_time = np.median(os_arr)
        surv_probs = cph.predict_survival_function(df, times=[median_time]).T.squeeze()
        brier = np.mean((surv_probs - (status_arr * (os_arr <= median_time))) ** 2)

        cox_res = {
            "cindex": float(cindex),
            "brier": float(brier),
            "hazard_ratios": cph.hazard_ratios_.to_dict(),
            "summary": cph.summary.to_dict(orient="list"),
        }
        results["cox"].append(cox_res)

    return results

In [ ]:
def transformation(scores):
    #return np.abs(scores)
    return scores

os = DATA["OS (corrected)"] / daysXmonth
status = DATA["status"] 

n_perms = 5000
n_comps = 100

order, stats = rank_components_by_survival(
    cmps_ucsf, os, status,
    transformation=transformation,   # your function
    n_components=n_comps,                 # e.g. first 100 components
    n_perms=n_perms,
)

results = evaluate_topk_models(
    cmps_ucsf, 
    os, 
    status, 
    range(n_comps),#order,
    max_k=n_comps,
    transformation=transformation,
    return_models=False
)

r2 = [r["r2"] for r in results["ols"]]
f_pvalue = [r["f_pvalue"] for r in results["ols"]]
r2_adj = [r["r2_adj"] for r in results["ols"]]
cindex = [r["cindex"] for r in results["cox"]]

fig, ax = plt.subplots(1, 4, figsize=(20,4))

ax[0].plot(r2)
ax[1].plot(np.log(f_pvalue))
ax[1].hlines(np.log(0.05), 0, 100, 'r', '--')
ax[2].plot(r2_adj)
ax[3].plot(cindex, 'o-', markersize=2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15,4))
ax[0].plot(stats["r"])
ax[1].plot(np.log(stats["p_perm"]))
ax[1].hlines(np.log(0.05), 0, 100, 'r', '--')
ax[2].scatter(np.log(stats["p_perm"][order].values), stats["r"][order].values)
ax[2].vlines(np.log(0.05), -.2, .1, 'r', '--')
ax[2].hlines(0,np.log(0.0005),np.log(0.05), color='gray', linestyle='--', linewidth=1)

In [ ]:
max(cindex), min(f_pvalue), max(r2)

In [ ]:
max(cindex), min(f_pvalue), max(r2)

In [ ]:
HRs = [r["summary"] for r in results["cox"]]

In [ ]:
K = 100
plt.plot(np.log(HRs[k]["p"]))
plt.hlines(np.log(0.05), 0, k, 'r', '--')

In [ ]:
HRs[2]

In [ ]:
K = 100
plt.plot(HRs[k]["coef"])